# Seed Variance and Final-Step Robustness

Goal: measure how stable each algorithm is across seeds and which methods have lower variance.

Expected takeaway: identify methods with strong final metrics and tight distributions.


In [ ]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

DATA_DIR_CANDIDATES = [
    Path.cwd() / "final_results" / "data",
    Path.cwd().parent / "final_results" / "data",
    Path.cwd() / "data",
    Path.cwd().parents[1] / "final_results" / "data",
    Path(__file__).resolve().parents[1] / "data" if '__file__' in globals() else Path.cwd() / "final_results" / "data",
]

# Robust data path resolver (works when running from repo root or final_results/tools)

# Robust data path resolver (works when running from repo root or final_results/tools)

def resolve_data_path(filename: str) -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "final_results" / "data" / filename,
        cwd / "data" / filename,
    ]
    for parent in [cwd] + list(cwd.parents):
        candidates.append(parent / "final_results" / "data" / filename)
        candidates.append(parent / "data" / filename)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename} in candidates: {candidates}")




combined_path = resolve_data_path("combined_wide.csv")
combined = pd.read_csv(combined_path)
combined = combined.drop(columns=[c for c in combined.columns if c.lower().startswith("unnamed")], errors="ignore")
combined["Step"] = pd.to_numeric(combined["Step"], errors="coerce")

metric_cols = ["avg@32", "pass@1", "pass@2", "pass@4", "pass@8", "pass@16", "pass@32", "pass@64", "pass@128"]
for c in metric_cols:
    combined[c] = pd.to_numeric(combined[c], errors="coerce")

passk_cols = [c for c in metric_cols if c.startswith("pass@")]
passk_ks = [int(c.split("@")[1]) for c in passk_cols]

final_step = int(combined["Step"].max())

# Aggregate across seeds for smoother curves
agg = combined.groupby(["Step", "algorithm"], as_index=False)[metric_cols].mean()

# Helpers
PLOT_COUNTER = 0

def new_fig(title: str):
    global PLOT_COUNTER
    PLOT_COUNTER += 1
    plt.figure()
    plt.title(title)


In [ ]:
final = combined[combined["Step"] == final_step].copy()


In [ ]:
# Plot 1: boxplots per metric at final step
for metric in metric_cols:
    new_fig(f"Final-step seed distribution for {metric}")
    sns.boxplot(data=final, x="algorithm", y=metric)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


In [ ]:
# Plot 2: mean ± std bar plots per metric
summary = final.groupby("algorithm")[metric_cols].agg(["mean", "std"]).reset_index()
for metric in metric_cols:
    sub = summary[["algorithm", (metric, "mean"), (metric, "std")]].copy()
    sub.columns = ["algorithm", "mean", "std"]
    new_fig(f"Final-step mean±std for {metric}")
    plt.bar(sub["algorithm"], sub["mean"], yerr=sub["std"], capsize=3)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


In [ ]:
# Plot 3: coefficient of variation heatmap (std / mean)
cv = summary.copy()
for metric in metric_cols:
    cv[(metric, "cv")] = cv[(metric, "std")] / cv[(metric, "mean")].replace(0, np.nan)
cv_cols = [col for col in cv.columns if isinstance(col, tuple) and col[1] == "cv"]
cv_matrix = cv[["algorithm"] + cv_cols].copy()
cv_matrix.columns = ["algorithm"] + [c[0] for c in cv_cols]

new_fig("Coefficient of variation by algorithm and metric")
heat = cv_matrix.set_index("algorithm")
sns.heatmap(heat, annot=False, cmap="viridis")
plt.tight_layout()
plt.show()


In [ ]:
# Plot 4: seed scatter for pass@1
new_fig("Seed scatter for pass@1 at final step")
sns.stripplot(data=final, x="algorithm", y="pass@1", jitter=0.15)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

print(f"Generated {PLOT_COUNTER} figures")
